## Scenario 1: A single data scientist participating in an ML competition

This scenario demonstrates how an individual data scientist can use MLflow to track machine learning experiments on their local machine. This is a common setup for solo projects, hackathons, or competitions where collaboration and remote access are not required.

### MLflow setup overview
- **Tracking server:** Not used (runs locally, no remote server)
- **Backend store:** Local filesystem (stores experiment metadata in the `mlruns/` folder)
- **Artifacts store:** Local filesystem (stores model files and other artifacts in the same `mlruns/` folder)

With this setup, all experiment runs, parameters, metrics, and artifacts are saved locally. You can explore and compare your experiments using the MLflow UI.

### How to use the MLflow UI
- You can launch the MLflow UI by running `mlflow ui` in your terminal, or by running the provided code cell in this notebook.
- The UI will be available at [http://localhost:5000](http://localhost:5000).
- Use the UI to browse experiments, compare runs, and inspect logged models and artifacts.

> **Tip:** This local setup is ideal for learning and prototyping. For team projects or production, you would typically use a remote tracking server and a more robust backend (e.g., a database and cloud storage).

In [1]:
import mlflow

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'file:///Users/Nachi%201/Library/CloudStorage/GoogleDrive-nachi%40student.ie.edu/My%20Drive/IE%20MBDS/Term%202/MLOps/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns'


In [3]:
mlflow.search_experiments()

/opt/anaconda3/envs/ie_mlops_new/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


[<Experiment: artifact_location='file:///Users/Nachi%201/Library/CloudStorage/GoogleDrive-nachi%40student.ie.edu/My%20Drive/IE%20MBDS/Term%202/MLOps/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/341524586137987579', creation_time=1772555971871, experiment_id='341524586137987579', last_update_time=1772555971871, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location='file:///Users/Nachi%201/Library/CloudStorage/GoogleDrive-nachi%40student.ie.edu/My%20Drive/IE%20MBDS/Term%202/MLOps/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/0', creation_time=1772555738816, experiment_id='0', last_update_time=1772555738816, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

### Creating an experiment and logging a new run

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import mlflow
import sklearn
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_experiment("my-experiment-1")

X, y = load_iris(return_X_y=True)
class_names = load_iris().target_names

models = [
    ("LogisticRegression", LogisticRegression(C=1.0, random_state=42, max_iter=1000, solver="lbfgs")),
    ("RandomForestClassifier", RandomForestClassifier(n_estimators=100, random_state=42))
]

for model_name, model in models:
    with mlflow.start_run() as run:
        mlflow.set_tag("model_type", model_name)
        mlflow.log_param("sklearn_version", sklearn.__version__)
        model.fit(X, y)
        y_pred = model.predict(X)
        acc = accuracy_score(y, y_pred)
        mlflow.log_metric("accuracy", acc)
        # Log confusion matrix as labeled DataFrame
        cm = confusion_matrix(y, y_pred)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
        cm_df.to_csv("confusion_matrix_labeled.csv")
        mlflow.log_artifact("confusion_matrix_labeled.csv")
        # Confusion matrix heatmap as image
        plt.figure(figsize=(5,4))
        sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Confusion Matrix ({model_name})")
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.savefig("confusion_matrix_heatmap.png")
        plt.close()
        mlflow.log_artifact("confusion_matrix_heatmap.png")
        # Provide input_example and use 'name' instead of deprecated 'artifact_path'
        input_example = np.expand_dims(X[0], axis=0)
        mlflow.sklearn.log_model(model, artifact_path="models", input_example=input_example)
        mlflow.set_tag("n_classes", len(np.unique(y)))
        mlflow.set_tag("run_time", datetime.datetime.now().isoformat())
        mlflow.set_tag("description", f"{model_name} on Iris dataset")
        print(f"Logged run for {model_name}, accuracy={acc:.3f}")

2026/03/03 18:00:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 18:00:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run for LogisticRegression, accuracy=0.973


2026/03/03 18:00:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 18:00:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run for RandomForestClassifier, accuracy=1.000


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///Users/Nachi%201/Library/CloudStorage/GoogleDrive-nachi%40student.ie.edu/My%20Drive/IE%20MBDS/Term%202/MLOps/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/341524586137987579', creation_time=1772555971871, experiment_id='341524586137987579', last_update_time=1772555971871, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location='file:///Users/Nachi%201/Library/CloudStorage/GoogleDrive-nachi%40student.ie.edu/My%20Drive/IE%20MBDS/Term%202/MLOps/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/0', creation_time=1772555738816, experiment_id='0', last_update_time=1772555738816, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [6]:
# Launch MLflow UI (default: uses local mlruns/ folder)
import subprocess
import sys

# This will run 'mlflow ui' as a background process
subprocess.Popen([sys.executable, '-m', 'mlflow', 'ui'])
print("MLflow UI started. Open http://localhost:5000 in your browser.")

MLflow UI started. Open http://localhost:5000 in your browser.


### Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [ ]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
except MlflowException:
    print("It's not possible to access the model registry :(")

/opt/anaconda3/envs/ie_mlops_new/lib/python3.10/site-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)


Backend store URI not provided. Using ./mlruns
Registry store URI not provided. Using backend store URI.


/opt/anaconda3/envs/ie_mlops_new/lib/python3.10/site-packages/mlflow/server/handlers.py:342: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, artifact_uri)
/opt/anaconda3/envs/ie_mlops_new/lib/python3.10/site-packages/mlflow/server/handlers.py:377: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connec

INFO:     127.0.0.1:53066 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:53066 - "GET /static-files/static/js/main.49592295.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53067 - "GET /static-files/static/css/main.e1790ccd.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:53066 - "GET /static-files/TelemetryLogger.telemetry-worker.e586432cbf500042667b.worker.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53066 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
INFO:     127.0.0.1:53070 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK
INFO:     127.0.0.1:53068 - "GET /static-files/favicon.ico HTTP/1.1" 200 OK
INFO:     127.0.0.1:53069 - "GET /static-files/manifest.json HTTP/1.1" 200 OK


/opt/anaconda3/envs/ie_mlops_new/lib/python3.10/site-packages/mlflow/server/handlers.py:342: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, artifact_uri)


...

/opt/anaconda3/envs/ie_mlops_new/lib/python3.10/site-packages/mlflow/server/handlers.py:377: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)


INFO:     127.0.0.1:53247 - "POST /api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
INFO:     127.0.0.1:53247 - "POST /api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
INFO:     127.0.0.1:53267 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=5&order_by=last_update_time+DESC HTTP/1.1" 200 OK
INFO:     127.0.0.1:53267 - "GET /static-files/static/js/8799.5c4e7066.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53273 - "GET /static-files/static/js/2365.07e09cf7.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53274 - "GET /static-files/static/css/6589.fd7db8ca.chunk.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:53275 - "GET /static-files/static/js/548.ba82e795.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53276 - "GET /static-files/static/js/6589.6e623b4c.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53274 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK


/opt/anaconda3/envs/ie_mlops_new/lib/python3.10/site-packages/mlflow/server/handlers.py:342: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, artifact_uri)
/opt/anaconda3/envs/ie_mlops_new/lib/python3.10/site-packages/mlflow/server/handlers.py:342: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, artifact_uri)
2026/03/03 18:02:26 ERROR mlflow.server.handlers: Error in _search_evaluation_datase

INFO:     127.0.0.1:53277 - "POST /graphql HTTP/1.1" 200 OK
INFO:     127.0.0.1:53267 - "GET /static-files/static/js/994.a4863046.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53273 - "GET /static-files/static/js/4445.0948167d.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53274 - "GET /static-files/static/js/9675.0cbf0e1e.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53276 - "GET /static-files/static/js/8983.a182a733.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53275 - "GET /static-files/static/js/2479.78e6486f.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53277 - "GET /static-files/static/js/1901.a7f9fdbb.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53267 - "GET /static-files/static/js/1664.1d0fa8d0.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53273 - "GET /static-files/static/js/6265.45977342.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53277 - "GET /static-files/static/js/2168.10d00de1.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:53273 - "POST /ajax-api/3.0/mlflow/traces/search

INFO:     Received SIGTERM, exiting.
INFO:     Received SIGTERM, exiting.
INFO:     Terminated child process [89667]
INFO:     Terminated child process [89668]
INFO:     Terminated child process [89669]
INFO:     Terminated child process [89670]
INFO:     Waiting for child process [89667]
INFO:     Shutting down
INFO:     Shutting down
INFO:     Shutting down
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [89669]
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [89668]
INFO:     Waiting for application shutdown.
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Application shutdown complete.
INFO:     Finished server process [89670]
INFO:     Finished server process [89667]
INFO:     Waiting for child process [89668]
INFO:     Waiting for child process [89669]
INFO:     Waiting 